# Nascenia — Bengali medical dialogue: train one chunk

Edit **CONFIG** below, then Run All. One pass trains **one arm per GPU in
parallel**, scores each on `val_dev`, and (optionally) writes a `submission.csv`.

On "GPU T4 x2" list two arms. Kaggle bills a session by wall-clock, not per-GPU,
so the second card is free compute — and Unsloth has no multi-GPU path, so two
independent runs is the only way to use it.

Everything is resumable. If the session dies, re-run the identical notebook:
`RESUME="auto"` picks up from the last checkpoint in `/kaggle/working`.
To continue on **new data** in a later session or a different account, set
`INIT_ADAPTER` to the previous adapter and `SKIP_ROWS` to the number the last
run printed.

**Rank arms on the `lex` column, not `score`.** Two constant-answer submissions
showed the organisers' BERTScore sits near 0.945 for any fluent Bengali medical
text, so half the metric is effectively a constant on the leaderboard while it
swings freely in our local composite. See `submissions/submissions.md`.

In [ ]:
# ============================== CONFIG ==============================
DATASET = "nascenia-bengali-med"     # your private dataset slug

# One arm per GPU. Kaggle bills a session by wall-clock, not per-GPU, so on
# "GPU T4 x2" the second card is free compute -- but only if an arm occupies it.
# name : hf-base : lora_r
ARMS = [
    "A_gemma2:unsloth/gemma-2-2b:32",
    "B_titulm:hishab/titulm-gemma-2-2b-v1.1:32",
]

MAX_ROWS  = 12000                    # rows PER ARM this session
SKIP_ROWS = 0                        # start this far into the fixed shuffle
MAX_SEQ   = 1024
LR        = 2e-4

# Effective batch is BATCH*ACCUM and must stay 16 -- it sets the step count and
# every LR number here is calibrated to it. Only the split changes.
#
# The ceiling is cross-entropy over Gemma's 256k vocab, not the model: without a
# fused loss, HF materialises a BATCH*MAX_SEQ*256000*4-byte logits tensor once per
# forward. That is 3.9 GB at BATCH=4 and it OOMs a 16 GB T4. BATCH=2 is 2.0 GB
# and fits. Turn USE_LIGER on and the tensor is never built at all, which is what
# lets BATCH go to 8 -- the canary reports peak GB either way, so believe it, not
# this comment.
BATCH = 2
ACCUM = 8
USE_LIGER = False                    # needs the pip install two cells down

MAX_HOURS = 10.5                     # stop + save before Kaggle's 12h kill
RESUME    = "auto"                   # "auto" = continue this run if a ckpt exists
INIT_ADAPTER = ""                    # e.g. /kaggle/input/prev-adapter/adapter

EVAL_ROWS  = 400                     # val rows for the loss number
SCORE_ROWS = 300                     # val_dev rows to generate + score
MAX_NEW    = 768
MAKE_SUBMISSION = False              # True once val_dev looks good
SUB_COLUMN = "output"                # settled: submission 001 was accepted with it
# ====================================================================

# Constant-answer floor. Not a target -- an arm below it has a broken loss mask.
BASELINE = 0.4574          # full composite on val_dev
BASELINE_LEX = 0.1020      # lexical half only  <-- rank arms on THIS
BASELINE_LB = 0.5743       # what that constant scored on the public leaderboard

# Two constant-answer submissions showed the organisers' BERTScore is ~0.945 for
# any fluent Bengali medical text, i.e. effectively constant. The real metric is
# LB ~= 0.473 + 0.3*TokenF1 + 0.2*ROUGE-L. Our local composite weights BERTScore
# at 0.5 where the leaderboard weights it at ~0, so the LEXICAL score is the
# honest ranking signal. See submissions/submissions.md.
assert BATCH * ACCUM == 16, f"effective batch must stay 16, got {BATCH*ACCUM}"
for a in ARMS:
    print(f"{a}   rows [{SKIP_ROWS}, {SKIP_ROWS+MAX_ROWS})  batch {BATCH}x{ACCUM}={BATCH*ACCUM}"
          f"  liger={USE_LIGER}")

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
from pathlib import Path

# NOTE: no CUDA_VISIBLE_DEVICES pin here -- train_arms.py pins one card per child
# process, which is what keeps HF Trainer from wrapping a 4-bit model in
# nn.DataParallel while still letting both T4s be used.

IN   = Path(f"/kaggle/input/{DATASET}")
ROOT = Path("/kaggle/working/ns")
assert IN.exists(), f"{IN} not found -- attach the dataset via '+ Add Input'"

ARM_NAMES = [a.split(":")[0] for a in ARMS]
ARM_BASES = {a.split(":")[0]: a.split(":")[1] for a in ARMS}

# Resolved now, not in the last cell: discovering the competition was never
# attached should cost 5 seconds, not a finished training run.
if MAKE_SUBMISSION:
    cands = [p for p in glob.glob("/kaggle/input/*/test.csv") if DATASET not in p]
    assert cands, ("MAKE_SUBMISSION=True but no test.csv -- attach the competition "
                   "via '+ Add Input -> Competitions'")
    TEST = cands[0]
    print(f"test inputs: {TEST}")

# Symlink the read-only inputs into a writable root so scripts can put the
# generated JSONL next to them without copying 100 MB of parquet.
(ROOT / "data").mkdir(parents=True, exist_ok=True)
for p in (IN / "data").glob("*"):
    dst = ROOT / "data" / p.name
    if not dst.exists():
        os.symlink(p, dst)
if (ROOT / "src").exists():
    shutil.rmtree(ROOT / "src")
shutil.copytree(IN / "src", ROOT / "src")

os.environ["NASCENIA_ROOT"] = str(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def sh(cmd):
    """Run a step with live output. Kaggle buffers a cell until it ends, so the
    -u / flush discipline in the scripts is what makes a 6h cell watchable."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True, text=True)
    if p.returncode:
        raise SystemExit(f"FAILED ({p.returncode}): {cmd}")

ARM_FLAGS = " ".join(f"--arm {a}" for a in ARMS)
print(ROOT, "->", sorted(x.name for x in (ROOT / "data").glob("*")))

In [ ]:
import torch
N_GPU = torch.cuda.device_count()
gpu   = torch.cuda.get_device_name(0)
bf16  = torch.cuda.is_bf16_supported()
print(f"{N_GPU} x {gpu} | bf16={bf16} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB each")

assert len(ARMS) <= N_GPU, (
    f"{len(ARMS)} arms but {N_GPU} GPU(s). Two arms on one card are slower than "
    f"running them back to back -- drop an arm, or pick 'GPU T4 x2' in Settings.")
if len(ARMS) < N_GPU:
    print(f"WARNING: {N_GPU - len(ARMS)} idle GPU(s). The session bills the same "
          f"either way -- add an arm and get the compute for free.")
if "P100" in gpu:
    print("WARNING: P100 has no tensor cores and is ~6x slower here. Prefer "
          "'GPU T4 x2' -- it is also two arms instead of one.")

# Unsloth is NOT installed here any more. The theory was that Gemma-2 overflows
# to inf in plain fp16 on T4, so Unsloth was mandatory. Measured on this stack it
# does not: the hf backend with --upcast-fp32 (which train_sft.py turns on
# automatically on fp16 hardware) trained for two hours at a finite, falling loss
# (1.03 -> 0.98). Unsloth also would not import on Kaggle's current image, so
# `--backend auto` silently fell through to hf regardless. Installing it cost
# minutes and bought a fallback that was already the thing running.
#
# What the hf backend does NOT give you is a fused loss, and Gemma's 256k vocab
# makes that the memory ceiling. Liger supplies it.
if USE_LIGER:
    print("installing liger-kernel (fused linear cross-entropy)")
    r = subprocess.run("pip install -q liger-kernel", shell=True, text=True,
                       capture_output=True)
    if r.returncode:
        raise SystemExit("liger install failed:\n" + (r.stderr or r.stdout)[-2000:])
    import importlib
    importlib.import_module("liger_kernel")
    print("liger-kernel ready")
else:
    print("liger off -> HF builds a "
          f"{BATCH}x{MAX_SEQ}x256000x4 = {BATCH*MAX_SEQ*256000*4/1e9:.1f} GB logits "
          f"tensor per forward. The canary will confirm it fits.")

import transformers, peft
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")

In [ ]:
# Canary: every arm, on its real GPU, at the real batch size, for 16 steps.
# It proves the whole risky surface at once -- OOM at this BATCH, fp16 nan loss,
# a tokenizer that will not load, Liger, and the two-process dual-GPU launch
# itself -- for about 5 minutes instead of 10 hours. Skipped when resuming.
#
# `Path.glob` returns a GENERATOR, and a generator object is always truthy. The
# obvious `any(dir.glob(...) for ...)` is therefore always True and silently
# skips the canary -- which is exactly how an OOM reached a 10-hour run once.
# `any()` must iterate the matches, so the inner glob has to be consumed.
RESUMING = RESUME == "auto" and any(
    any((ROOT / "runs" / n).glob("checkpoint-*")) for n in ARM_NAMES)
print(f"resuming: {RESUMING}")

if RESUMING:
    print("checkpoint present -> skipping canary")
else:
    sh(f"python -u src/train_arms.py {ARM_FLAGS} "
       f"--max-rows {16*BATCH*ACCUM} --eval-rows 16 --max-seq-len {MAX_SEQ} "
       f"--batch-size {BATCH} --grad-accum {ACCUM} --use-liger {int(USE_LIGER)} "
       f"--save-fraction 0 --eval-fraction 0 --progress-every 30 --poll-every 30")

    for n in ARM_NAMES:
        m = json.loads((ROOT / "runs" / n / "train_metrics.json").read_text())
        assert m["train_loss"] == m["train_loss"], f"{n}: nan loss -- fp16 overflow"
        est = m["hours_per_full_epoch"] * MAX_ROWS / 90508
        print(f"{n:<12} loss={m['train_loss']:.3f}  peak={m['peak_gpu_gb']} GB  "
              f"{m['tokens_per_second']:.0f} tok/s  -> {MAX_ROWS} rows in {est:.1f} h")
        if m["peak_gpu_gb"] < 11 and BATCH < 8:
            print(f"             headroom left: try BATCH={BATCH*2}, ACCUM={ACCUM//2}")
        if est > MAX_HOURS:
            fits = int(MAX_ROWS * MAX_HOURS / est / 1000) * 1000
            print(f"             !! over MAX_HOURS={MAX_HOURS}. It would stop early and "
                  f"save, but a cosine frozen mid-anneal at a high LR is worse than a "
                  f"completed shorter schedule. Set MAX_ROWS={fits}, or plan a second "
                  f"session with RESUME='auto' to finish the same schedule.")

    # The canary trained real (tiny) adapters into the real run dirs. Clear them
    # or `--resume auto` in the next cell would continue a 16-step schedule.
    for n in ARM_NAMES:
        shutil.rmtree(ROOT / "runs" / n, ignore_errors=True)
    print("\ncanary run dirs cleared")

In [ ]:
# ============================ TRAIN ============================
# One arm per GPU, in parallel. A status table for every arm prints every 2 min;
# runs/<name>/progress.json carries the same numbers, and runs/<name>/train.log
# has each arm's full output.
#
# The ETA is elapsed/fraction-done, so it includes model load, tokenisation and
# evals. tqdm's ETA inside train.log is the recent per-step rate only and reads
# lower. The status table is the one to trust for "will this finish".
opts = (f"{ARM_FLAGS} "
        f"--max-rows {MAX_ROWS} --skip-rows {SKIP_ROWS} --eval-rows {EVAL_ROWS} "
        f"--max-seq-len {MAX_SEQ} --lr {LR} "
        f"--batch-size {BATCH} --grad-accum {ACCUM} --use-liger {int(USE_LIGER)} "
        f"--max-hours {MAX_HOURS} --progress-every 120 --poll-every 120 "
        f"--save-fraction 0.25 --eval-fraction 0.5")
if INIT_ADAPTER:
    opts += f" --init-adapter {INIT_ADAPTER}"
elif RESUME:
    opts += f" --resume {RESUME}"

sh(f"python -u src/train_arms.py {opts}")

STATE = {n: json.loads((ROOT / "runs" / n / "chunk_state.json").read_text())
         for n in ARM_NAMES}
print("\n" + json.dumps(STATE, indent=2))
for n, s in STATE.items():
    if not s["finished"]:
        print(f"\n!! {n} hit MAX_HOURS. Re-run this notebook unchanged to continue "
              f"(RESUME='auto').")

In [ ]:
# Generate on val_dev and score each arm. Sequential: generation is ~10 min per
# arm and both cards are free now, but a second launcher for a 20-minute job is
# complexity that buys nothing.
for n in ARM_NAMES:
    RUN = ROOT / "runs" / n
    sh(f"python -u src/generate.py --base-model {ARM_BASES[n]} --adapter {RUN}/adapter "
       f"--data data/val_dev.parquet --limit {SCORE_ROWS} --batch-size 8 "
       f"--max-new-tokens {MAX_NEW} --out {RUN}/val_dev.parquet")
    sh(f"python -u src/score_run.py --pred {RUN}/val_dev.parquet --ref data/val_dev.parquet "
       f"--subset-ok --tag {n}@greedy "
       f"--notes 'rows {SKIP_ROWS}-{SKIP_ROWS+MAX_ROWS} b{BATCH}x{ACCUM}'")

In [ ]:
import pandas as pd
res = pd.read_csv(ROOT / "runs" / "results.csv")
mine = res[res["tag"].str.split("@").str[0].isin(ARM_NAMES)].copy()

# The ranking column. Two constant-answer submissions put the organisers'
# BERTScore at ~0.945 regardless of content, so the composite's 0.5*BERTScore
# term is nearly constant on the leaderboard while it swings freely here. Rank
# on lexical; keep the composite visible only as a sanity check.
mine["lex"] = 0.3 * mine["token_f1"] + 0.2 * mine["rouge_l_f1"]
mine["LB_est"] = 0.4725 + mine["lex"]
mine = mine.sort_values("lex", ascending=False)

print(mine[["tag", "n", "lex", "LB_est", "score", "bertscore_f1", "token_f1",
            "rouge_l_f1", "pred_words_mean"]].to_string(index=False))
print(f"\nconstant-answer floor: lex {BASELINE_LEX:.4f} | composite {BASELINE:.4f} "
      f"| actual LB {BASELINE_LB:.4f}")
print("references average 102 words -- both lexical metrics are F1s, so "
      "overshooting costs precision and undershooting costs recall.")

for _, r in mine.iterrows():
    verdict = "OK" if r["lex"] > BASELINE_LEX else "BROKEN, do not submit"
    print(f"  {r['tag']:<20} lex {r['lex']:.4f}  -> {verdict}")

WINNER = mine.iloc[0]["tag"].split("@")[0]
print(f"\nwinner: {WINNER}")

# Read a few by hand. Phase 2 judges medical accuracy, and a model that has
# learned the boilerplate but says nothing clinical scores well here and badly there.
gen = pd.read_parquet(ROOT / "runs" / WINNER / "val_dev.parquet")
for i in range(3):
    print(f"\n--- {gen['id'][i]} ---\n{gen['output'][i][:600]}")

In [ ]:
# Parameter count on the MERGED winner -- this is what Phase 2 verifies. On CPU,
# so it does not fight the GPUs, and only for the arm that will actually ship.
import torch, gc
from transformers import AutoModelForCausalLM
from peft import PeftModel

RUN = ROOT / "runs" / WINNER
base = AutoModelForCausalLM.from_pretrained(ARM_BASES[WINNER], dtype=torch.bfloat16,
                                            attn_implementation="eager", device_map="cpu")
merged = PeftModel.from_pretrained(base, f"{RUN}/adapter").merge_and_unload()
n = sum(p.numel() for p in merged.parameters())
print(f"{WINNER} ({ARM_BASES[WINNER]}) merged params: {n:,} = {n/1e9:.4f}B")
assert n <= 3_000_000_000, "OVER THE 3B CAP -- disqualified at Phase 2"
print("under the 3B cap")
(RUN / "param_count.txt").write_text(f"{ARM_BASES[WINNER]}\n{n}\n{n/1e9:.4f}B\n")
del base, merged; gc.collect(); torch.cuda.empty_cache()

In [ ]:
if MAKE_SUBMISSION:
    RUN = ROOT / "runs" / WINNER
    sh(f"python -u src/generate.py --base-model {ARM_BASES[WINNER]} --adapter {RUN}/adapter "
       f"--data {TEST} --batch-size 8 --max-new-tokens {MAX_NEW} "
       f"--out {RUN}/test_preds.parquet")
    sh(f"python -u src/make_submission.py --pred {RUN}/test_preds.parquet --test {TEST} "
       f"--out /kaggle/working/submission.csv --column {SUB_COLUMN}")
    print(f"\nSubmit from the notebook's Output tab, or download submission.csv.")
    print(f"Then archive it as submissions/00N_{WINNER}.csv and add a row to "
          f"submissions/submissions.md WITH the local lexical score -- a submission "
          f"whose local number was never recorded cannot be used to calibrate.")
else:
    print("MAKE_SUBMISSION=False -- val_dev only.")

In [ ]:
# Keep every arm's adapter + logs, drop the optimizer checkpoints unless that arm
# stopped early and needs to be resumed. Kaggle caps notebook output at 20 GB and
# two arms' checkpoints will reach it.
OUT = Path("/kaggle/working/out"); OUT.mkdir(exist_ok=True)
for n in ARM_NAMES:
    RUN = ROOT / "runs" / n
    dst = OUT / n
    shutil.copytree(RUN / "adapter", dst / "adapter", dirs_exist_ok=True)
    for f in ("train_config.json", "train_metrics.json", "chunk_state.json",
              "progress.json", "param_count.txt", "train.log"):
        if (RUN / f).exists():
            shutil.copy2(RUN / f, dst / f)
    if STATE[n]["finished"]:
        for c in RUN.glob("checkpoint-*"):
            shutil.rmtree(c)
    print(f"{n:<12} adapter saved, checkpoints kept: {not STATE[n]['finished']}")

shutil.copy2(ROOT / "runs" / "results.csv", OUT / "results.csv")
for p in (ROOT / "data").glob("sft_*.jsonl"):
    p.unlink()

sh("du -sh /kaggle/working/* | sort -h")
print("\nDownload /kaggle/working/out/<arm>/adapter, or 'Save Version' and add this\n"
      "notebook's output as an input to the next chunk.")
for n in ARM_NAMES:
    print(f"NEXT CHUNK for {n}:  SKIP_ROWS={STATE[n]['next_skip_rows']}  "
          f"INIT_ADAPTER=/kaggle/input/<this-output>/out/{n}/adapter")